## Install Dependencies

In [1]:
# Install latest compatible dependencies 
!pip install -q transformers>=4.46.0 trl>=0.12.0 peft>=0.13.0 bitsandbytes>=0.44.0 datasets>=3.0.0 accelerate>=1.0.0 wandb textstat rouge-score sentencepiece

## Imports & Configuration

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch
import wandb
import textstat
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model, PeftModel
from trl import SFTConfig, SFTTrainer
from rouge_score import rouge_scorer

# ─── Kaggle Secrets ───────────────────────────────────────────────────────────
# On Kaggle, use the Secrets add-on (Add-ons → Secrets) to store HF_TOKEN and
# WANDB_API_KEY, then attach them to the notebook.
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret("HF_TOKEN")
WANDB_API_KEY = secrets.get_secret("WANDB_API_KEY")

os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["WANDB_API_KEY"] = WANDB_API_KEY
os.environ["WANDB_PROJECT"] = "llama3-medical-simplification"

# ─── Constants ────────────────────────────────────────────────────────────────
MODEL_ID = "meta-llama/Llama-3.2-1B-Instruct"
NEW_MODEL_NAME = "llama-3.2-1b-medical-simplifier"  # Hub repo name
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16  # T4 does NOT support bf16

print(f"🔧 Device : {DEVICE}")
print(f"🔧 GPU    : {torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'N/A'}")
print(f"🔧 Dtype  : {DTYPE}")
print(f"🔧 Visible GPUs: {torch.cuda.device_count()}")  # Should print 1

# Initialize W&B
wandb.login(key=WANDB_API_KEY)
wandb.init(
    project="llama3-medical-simplification",
    name="qlora-llama3.2-1b-plos",
    config={
        "model": MODEL_ID,
        "quantization": "4-bit NF4",
        "compute_dtype": "float16",
        "lora_r": 16,
        "lora_alpha": 32,
    },
)

🔧 Device : cuda
🔧 GPU    : Tesla T4
🔧 Dtype  : torch.float16
🔧 Visible GPUs: 1


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: zeeshanwarraich51 (zeeshanwarraich51-the-university-of-lahore) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## Load Model & Tokenizer with QLoRA (4-bit, fp16)

In [3]:
# ─── BitsAndBytes 4-bit config ────────────────────────────────────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

# ─── Tokenizer ───────────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN,
    trust_remote_code=False,
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# ─── Model ───────────────────────────────────────────────────────────────────
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map={"": 0},
    torch_dtype=torch.float16,
    token=HF_TOKEN,
    trust_remote_code=False,
)
model.config.use_cache = False

# ─── LoRA adapters ───────────────────────────────────────────────────────────
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
)

model = get_peft_model(model, lora_config)

# ─── Cast any stray bfloat16 params to float16 (Llama 3 defaults to bf16) ────
import bitsandbytes as bnb
model.config.torch_dtype = torch.float16
for name, param in model.named_parameters():
    if param.dtype == torch.bfloat16:
        param.data = param.data.to(torch.float16)

for name, module in model.named_modules():
    if isinstance(module, bnb.nn.Linear4bit):
        module.compute_dtype = torch.float16
    if 'lora_' in name:
        module.to(torch.float16)

model.print_trainable_parameters()


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

trainable params: 3,407,872 || all params: 1,239,222,272 || trainable%: 0.2750


## Load Dataset & Format with Llama 3 Chat Template

In [4]:
# ─── Load the Parquet-safe dataset (no trust_remote_code needed) ─────────────
dataset = load_dataset("pszemraj/scientific_lay_summarisation-plos-norm")
print(f"📦 Train : {len(dataset['train'])} samples")
print(f"📦 Valid : {len(dataset['validation'])} samples")
print(f"📦 Test  : {len(dataset['test'])} samples")

# ─── Inspect column names ───────────────────────────────────────────────────
print(f"📋 Columns: {dataset['train'].column_names}")

# ─── System prompt ───────────────────────────────────────────────────────────
SYSTEM_PROMPT = (
    "You are a medical text simplifier. Your job is to rewrite complex medical "
    "and scientific text so that a 5th grader can easily understand it. Use "
    "simple words, short sentences, and explain any technical terms. Keep the "
    "key facts accurate."
)

def format_chat(example):
    """
    Format a single example using the Llama 3 Chat Template.
    
    Expected dataset columns: 'article' (technical) and 'summary' (lay summary).
    """
    technical_text = example.get("article", example.get("input", ""))
    simple_text = example.get("summary", example.get("target", ""))
    
    # Truncate very long articles to keep within context limits (speedup + memory optimization)
    max_input_chars = 1200
    if len(technical_text) > max_input_chars:
        technical_text = technical_text[:max_input_chars] + "..."
    
    # Llama 3 Chat Template
    chat = (
        "<|begin_of_text|>"
        "<|start_header_id|>system<|end_header_id|>\n\n"
        f"{SYSTEM_PROMPT}<|eot_id|>"
        "<|start_header_id|>user<|end_header_id|>\n\n"
        f"Simplify the following medical text:\n\n{technical_text}<|eot_id|>"
        "<|start_header_id|>assistant<|end_header_id|>\n\n"
        f"{simple_text}<|eot_id|>"
    )
    
    return {"text": chat}

# Apply formatting
train_dataset = dataset["train"].map(format_chat, remove_columns=dataset["train"].column_names)
eval_dataset = dataset["validation"].map(format_chat, remove_columns=dataset["validation"].column_names)
test_dataset = dataset["test"].map(format_chat, remove_columns=dataset["test"].column_names)

print(f"\n✅ Formatted {len(train_dataset)} training examples")
print(f"📝 Sample:\n{train_dataset[0]['text'][:500]}...")


📦 Train : 24773 samples
📦 Valid : 1376 samples
📦 Test  : 1376 samples
📋 Columns: ['article', 'summary', 'section_headings', 'keywords', 'year', 'title', 'article_length', 'summary_length']

✅ Formatted 24773 training examples
📝 Sample:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a medical text simplifier. Your job is to rewrite complex medical and scientific text so that a 5th grader can easily understand it. Use simple words, short sentences, and explain any technical terms. Keep the key facts accurate.<|eot_id|><|start_header_id|>user<|end_header_id|>

Simplify the following medical text:

Kidney function depends on the nephron, which comprises a blood filter, a tubule that is subdivided into functio...


## SFTConfig & SFTTrainer (TRL 0.12+ API)

In [5]:
# ─── SFTConfig (replaces TrainingArguments in TRL 0.12+) ─────────────────────
sft_config = SFTConfig(
    output_dir="./results",
    
    # ── Training hyperparameters ─────────────────────────────────────────
    # Using max_steps instead of full epochs to drastically speed up training (~30-40 min vs 4 hours)
    max_steps=300,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,  # Reduced from 2e-4 to prevent NaN loss & mode collapse
    weight_decay=0.01,
    warmup_steps=30,  # 10% of 300 max_steps
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    
    # ── Precision ────────────────────────────────────────────────────────
    # DO NOT set fp16=True — it enables the GradScaler which crashes on T4
    # because bitsandbytes internally produces bf16 gradients during
    # dequantization.  The model already computes in fp16 natively via
    # bnb_4bit_compute_dtype=float16, so training is still efficient.
    fp16=False,
    bf16=False,
    
    # ── Logging & evaluation ─────────────────────────────────────────────
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    
    # ── SFT-specific fields (MUST be inside SFTConfig, not in Trainer) ──
    dataset_text_field="text",
    max_length=512,  # Reduced from 1024 for speed and T4 memory stability
    
    # ── W&B integration ──────────────────────────────────────────────────
    report_to="wandb",
    run_name="qlora-llama3.2-1b-plos-fast",
    
    # ── Misc ─────────────────────────────────────────────────────────────
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_grad_norm=0.1,  # Strict clipping to prevent exploding loss (NaN)
    seed=42,
    
    # ── CRITICAL: Prevent DataParallel on multi-GPU Kaggle setups ─────────
    # Even with CUDA_VISIBLE_DEVICES=0, explicitly disable parallelism
    ddp_find_unused_parameters=None,
)

# ─── SFTTrainer ──────────────────────────────────────────────────────────────
# TRL 0.12+: use `processing_class` instead of `tokenizer`
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,  # NOT `tokenizer=tokenizer`
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False, pad_to_multiple_of=8),
)

print("✅ Trainer ready")


✅ Trainer ready


## Train 🚀

In [6]:
print("🚀 Starting training...")
trainer.train()

# Save final checkpoint
trainer.save_model("./final_model")
tokenizer.save_pretrained("./final_model")
print("✅ Training complete! Model saved to ./final_model")


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


🚀 Starting training...


Step,Training Loss,Validation Loss
50,2.369613,2.311313
100,2.226883,2.221278
150,2.217078,2.212246
200,2.158298,2.200933
250,2.176056,2.198545
300,2.200377,2.198430


✅ Training complete! Model saved to ./final_model


## Evaluation: ROUGE-L & Flesch-Kincaid Grade Level

In [7]:
def generate_simplification(model, tokenizer, technical_text, max_new_tokens=512):
    """Generate a simplified version of the given technical text."""
    prompt = (
        "<|begin_of_text|>"
        "<|start_header_id|>system<|end_header_id|>\n\n"
        f"{SYSTEM_PROMPT}<|eot_id|>"
        "<|start_header_id|>user<|end_header_id|>\n\n"
        f"Simplify the following medical text:\n\n{technical_text}<|eot_id|>"
        "<|start_header_id|>assistant<|end_header_id|>\n\n"
    )
    
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=768).to(DEVICE)
    
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    # Decode only the generated part (skip the prompt tokens)
    generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True)


def evaluate_model(model, tokenizer, test_dataset, num_samples=20):
    """
    Evaluate on a subset of test examples:
      - ROUGE-L for semantic similarity
      - Flesch-Kincaid Grade Level for readability
    """
    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    
    rouge_l_scores = []
    fk_original_scores = []
    fk_simplified_scores = []
    
    # Reload raw test set for access to original columns
    raw_test = load_dataset("pszemraj/scientific_lay_summarisation-plos-norm", split="test")
    
    sample_indices = np.random.choice(len(raw_test), size=min(num_samples, len(raw_test)), replace=False)
    
    print(f"\n📊 Evaluating on {len(sample_indices)} test samples...\n")
    
    for i, idx in enumerate(sample_indices):
        example = raw_test[int(idx)]
        technical_text = example.get("article", example.get("input", ""))[:2000]
        reference = example.get("summary", example.get("target", ""))
        
        # Generate simplification
        prediction = generate_simplification(model, tokenizer, technical_text)
        
        # ROUGE-L
        rouge_result = scorer.score(reference, prediction)
        rouge_l = rouge_result["rougeL"].fmeasure
        rouge_l_scores.append(rouge_l)
        
        # Flesch-Kincaid Grade Level
        fk_orig = textstat.flesch_kincaid_grade(technical_text)
        fk_simplified = textstat.flesch_kincaid_grade(prediction) if len(prediction.split()) > 5 else float("nan")
        fk_original_scores.append(fk_orig)
        if not np.isnan(fk_simplified):
            fk_simplified_scores.append(fk_simplified)
        
        if i < 3:  # Print first 3 examples
            print(f"── Example {i+1} ──")
            print(f"  📄 Original FK Grade  : {fk_orig:.1f}")
            print(f"  ✏️  Simplified FK Grade: {fk_simplified:.1f}")
            print(f"  📏 ROUGE-L            : {rouge_l:.4f}")
            print(f"  📝 Prediction (first 200 chars): {prediction[:200]}...")
            print()
    
    # Aggregate metrics
    avg_rouge_l = np.mean(rouge_l_scores)
    avg_fk_original = np.mean(fk_original_scores)
    avg_fk_simplified = np.mean(fk_simplified_scores) if fk_simplified_scores else float("nan")
    fk_improvement = avg_fk_original - avg_fk_simplified
    
    print("=" * 60)
    print("📊 EVALUATION RESULTS")
    print("=" * 60)
    print(f"  ROUGE-L (avg)                    : {avg_rouge_l:.4f}")
    print(f"  Flesch-Kincaid Original (avg)     : {avg_fk_original:.1f}")
    print(f"  Flesch-Kincaid Simplified (avg)   : {avg_fk_simplified:.1f}")
    print(f"  FK Grade Improvement              : {fk_improvement:+.1f} levels")
    print("=" * 60)
    
    # Log to W&B
    wandb.log({
        "eval/rouge_l": avg_rouge_l,
        "eval/fk_original": avg_fk_original,
        "eval/fk_simplified": avg_fk_simplified,
        "eval/fk_improvement": fk_improvement,
    })
    
    return {
        "rouge_l": avg_rouge_l,
        "fk_original": avg_fk_original,
        "fk_simplified": avg_fk_simplified,
        "fk_improvement": fk_improvement,
    }

# Run evaluation
eval_results = evaluate_model(model, tokenizer, test_dataset)



📊 Evaluating on 20 test samples...

── Example 1 ──
  📄 Original FK Grade  : 15.9
  ✏️  Simplified FK Grade: 16.0
  📏 ROUGE-L            : 0.3128
  📝 Prediction (first 200 chars): Nitrogen is the most common limiting nutrient in soil and the primary source of nitrogen for the world's growing population. Most biological nitrogen fixation is catalyzed by molybdenum-dependent nitr...

── Example 2 ──
  📄 Original FK Grade  : 17.5
  ✏️  Simplified FK Grade: 17.3
  📏 ROUGE-L            : 0.2415
  📝 Prediction (first 200 chars): Actin filaments play a critical role in the development and function of many cell types, including yeast cells, and are involved in a wide range of cellular processes. However, the dynamics of actin p...

── Example 3 ──
  📄 Original FK Grade  : 14.7
  ✏️  Simplified FK Grade: 13.3
  📏 ROUGE-L            : 0.2208
  📝 Prediction (first 200 chars): Rift Valley Fever (RVF) is a zoonotic disease caused by RVF virus (RVFV), which is transmitted to humans by Aedes and Cul

## Push LoRA Adapters & Tokenizer to Hugging Face Hub

In [8]:
print(f"📤 Pushing model to Hugging Face Hub as: {NEW_MODEL_NAME}")

# Push the LoRA adapters
model.push_to_hub(
    NEW_MODEL_NAME,
    token=HF_TOKEN,
    commit_message="Upload QLoRA adapters for medical text simplification",
    private=True,
)

# Push the tokenizer
tokenizer.push_to_hub(
    NEW_MODEL_NAME,
    token=HF_TOKEN,
    commit_message="Upload tokenizer",
)

print(f"✅ Model and tokenizer pushed to: https://huggingface.co/{NEW_MODEL_NAME}")


# %%
def catastrophic_forgetting_check(model, tokenizer):
    """
    Ask the model a few general-knowledge questions that have nothing to do
    with medical text simplification.  If the answers are coherent, the model
    has not catastrophically forgotten its pre-trained knowledge.
    """
    general_questions = [
        "What is the capital of France?",
        "Explain photosynthesis in one sentence.",
        "Who wrote the play 'Romeo and Juliet'?",
        "What is 15 multiplied by 7?",
        "Name three planets in our solar system.",
    ]
    
    print("=" * 60)
    print("🛡️  CATASTROPHIC FORGETTING CHECK")
    print("=" * 60)
    
    results = []
    for q in general_questions:
        prompt = (
            "<|begin_of_text|>"
            "<|start_header_id|>user<|end_header_id|>\n\n"
            f"{q}<|eot_id|>"
            "<|start_header_id|>assistant<|end_header_id|>\n\n"
        )
        
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=256).to(DEVICE)
        
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=100,
                temperature=0.3,
                top_p=0.9,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
            )
        
        generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]
        answer = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
        
        results.append({"question": q, "answer": answer})
        print(f"\n❓ Q: {q}")
        print(f"💬 A: {answer}")
    
    print("\n" + "=" * 60)
    print("✅ Forgetting check complete — review answers above for coherence.")
    print("=" * 60)
    
    # Log to W&B as a table
    forgetting_table = wandb.Table(columns=["Question", "Answer"])
    for r in results:
        forgetting_table.add_data(r["question"], r["answer"])
    wandb.log({"catastrophic_forgetting_check": forgetting_table})
    
    return results

forgetting_results = catastrophic_forgetting_check(model, tokenizer)

# %%
# Finish W&B run
wandb.finish()
print("🎉 All done! W&B run finalized.")
print(f"📦 LoRA adapters available at: https://huggingface.co/{NEW_MODEL_NAME}")


📤 Pushing model to Hugging Face Hub as: llama-3.2-1b-medical-simplifier


README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


✅ Model and tokenizer pushed to: https://huggingface.co/llama-3.2-1b-medical-simplifier
🛡️  CATASTROPHIC FORGETTING CHECK

❓ Q: What is the capital of France?
💬 A: The capital of France is Paris.

❓ Q: Explain photosynthesis in one sentence.
💬 A: Photosynthesis is the process by which plants, algae, and some bacteria convert light energy from the sun into chemical energy in the form of organic compounds, such as glucose, using water and carbon dioxide.

❓ Q: Who wrote the play 'Romeo and Juliet'?
💬 A: The play "Romeo and Juliet" was written by William Shakespeare.

❓ Q: What is 15 multiplied by 7?
💬 A: 15 multiplied by 7 is 105.

❓ Q: Name three planets in our solar system.
💬 A: Mercury, Mars, and Jupiter are three planets in our solar system.

✅ Forgetting check complete — review answers above for coherence.


eval/entropy,█▃▂▁▁▁
eval/fk_improvement,▁
eval/fk_original,▁
eval/fk_simplified,▁
eval/loss,█▂▂▁▁▁
eval/mean_token_accuracy,▁▇████
eval/num_tokens,▁▂▄▅▇█
eval/rouge_l,▁
eval/runtime,█▁▁▆▆▃
eval/samples_per_second,▁██▃▃▆
+9,...


🎉 All done! W&B run finalized.
📦 LoRA adapters available at: https://huggingface.co/llama-3.2-1b-medical-simplifier
